# Week 5 – Mid-Review: Data Audit & Initial Statistical Findings
## Ames Housing Dataset Analysis

**Course:** 24ADI204 – Data Science & Visualization  
**Team:** Team 8  
**Academic Year:** 2026  

---

## 1. Introduction

The main objectives of **Week 5** are:
1. Perform a comprehensive **Data Audit** of the cleaned Ames Housing dataset produced during the Week 3 cleaning sprint.
2. Verify data quality, integrity, and confirm the absence of unintended missing values or duplicate records.
3. Calculate descriptive statistics (central tendency, dispersion, quartiles, and IQR) for key numerical variables.
4. Compare mean vs. median to evaluate distribution symmetry and the impact of right-skewness and extreme values.
5. Analyze feature variability and skewness across major structural attributes.
6. Compute Pearson correlation coefficients to identify top numerical predictors of `SalePrice`.
7. Provide key statistical evidence and empirical findings that establish a foundation for Week 6 Feature Engineering.

In [1]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (10, 6)
plt.rcParams["font.sans-serif"] = "DejaVu Sans"

print("Environment loaded successfully. Pandas version:", pd.__version__)

Environment loaded successfully. Pandas version: 3.0.1


## 2. Load the Cleaned Dataset

We reproduce the exact cleaning methodology established in Week 3 on `AmesHousing_Uncleaned.csv` to ensure total reproducibility, saving the result as `Cleaned_AmesHousing_Week5.csv`.

In [2]:
data_path = "../Week 3/AmesHousing_Uncleaned.csv"
df_raw = pd.read_csv(data_path)
df = df_raw.copy()

# Week 3 Imputation Logic
no_feature_cols = [
    "Alley", "Mas Vnr Type", "Bsmt Qual", "Bsmt Cond", "Bsmt Exposure",
    "BsmtFin Type 1", "BsmtFin Type 2", "Fireplace Qu", "Garage Type",
    "Garage Finish", "Garage Qual", "Garage Cond", "Pool QC", "Fence",
    "Misc Feature"
]
for col in no_feature_cols:
    if col in df.columns:
        df[col] = df[col].fillna("None")

zero_if_absent = [
    "Mas Vnr Area", "BsmtFin SF 1", "BsmtFin SF 2", "Bsmt Unf SF",
    "Total Bsmt SF", "Bsmt Full Bath", "Bsmt Half Bath", "Garage Cars",
    "Garage Area"
]
for col in zero_if_absent:
    if col in df.columns:
        df[col] = df[col].fillna(0)

if "Lot Frontage" in df.columns:
    df["Lot Frontage"] = df["Lot Frontage"].fillna(df["Lot Frontage"].median())

if "Garage Yr Blt" in df.columns:
    df["Garage Yr Blt"] = df["Garage Yr Blt"].fillna(0)

numeric_cols = df.select_dtypes(include=np.number).columns
for col in numeric_cols:
    if df[col].isna().any():
        df[col] = df[col].fillna(df[col].median())

categorical_cols = df.select_dtypes(include=["object", "string"]).columns
for col in categorical_cols:
    if df[col].isna().any():
        mode = df[col].mode()
        if not mode.empty:
            df[col] = df[col].fillna(mode.iloc[0])

df.to_csv("Cleaned_AmesHousing_Week5.csv", index=False)
print("Cleaned dataset loaded successfully.")
print(f"Dataset Shape: {df.shape}")

Cleaned dataset loaded successfully.
Dataset Shape: (2930, 82)


## 3. Dataset Overview

Summary of dataset dimensions, data types, and initial records.

In [3]:
print("Total Rows:", df.shape[0])
print("Total Columns:", df.shape[1])
num_cols = df.select_dtypes(include=np.number).columns
cat_cols = df.select_dtypes(include=["object", "string"]).columns
print("Numerical Columns:", len(num_cols))
print("Categorical Columns:", len(cat_cols))
display(df.head())

Total Rows: 2930
Total Columns: 82
Numerical Columns: 39
Categorical Columns: 43


## 4. Final Data Audit

A summary table validating overall data quality.

In [4]:
audit_table = pd.DataFrame({
    "Audit Metric": ["Total Rows", "Total Columns", "Duplicate Rows", "Missing Values", "Numerical Features", "Categorical Features"],
    "Result": [df.shape[0], df.shape[1], df.duplicated().sum(), df.isna().sum().sum(), len(num_cols), len(cat_cols)]
})
display(audit_table)

## 5. Missing Value Analysis

Verifying that all missing values have been appropriately handled.

In [5]:
rem_missing = df.isna().sum().sum()
print(f"Remaining missing values in cleaned dataset: {rem_missing}")

Remaining missing values in cleaned dataset: 0


## 6. Descriptive Statistics

Calculating central tendency, dispersion, quartiles, and IQR for key housing features.

In [6]:
key_vars = ["SalePrice", "Gr Liv Area", "Overall Qual", "Year Built", "Total Bsmt SF", "Garage Cars", "Garage Area", "1st Flr SF"]
desc_list = []
for var in key_vars:
    s = df[var]
    q1 = s.quantile(0.25)
    q3 = s.quantile(0.75)
    desc_list.append({
        "Feature": var,
        "Mean": round(s.mean(), 2),
        "Std Dev": round(s.std(), 2),
        "Min": s.min(),
        "Q1 (25%)": q1,
        "Median": s.median(),
        "Q3 (75%)": q3,
        "Max": s.max(),
        "IQR": round(q3 - q1, 2),
        "Skewness": round(s.skew(), 2)
    })
desc_df = pd.DataFrame(desc_list)
display(desc_df)

## 7. Mean vs. Median Analysis

Evaluating central tendency differences to detect distribution skewness. For instance, `SalePrice` has a mean of **$180,796.06** and a median of **$160,000.00** (mean > median by **$20,796.06**), demonstrating strong positive (right) skewness due to high-value property sales.

In [7]:
mm_comp = pd.DataFrame({
    "Feature": key_vars,
    "Mean": [round(df[v].mean(), 2) for v in key_vars],
    "Median": [round(df[v].median(), 2) for v in key_vars],
    "Difference (Mean - Median)": [round(df[v].mean() - df[v].median(), 2) for v in key_vars],
    "Distribution Pattern": ["Right-Skewed" if df[v].mean() > df[v].median() else ("Left-Skewed" if df[v].mean() < df[v].median() else "Symmetric") for v in key_vars]
})
display(mm_comp)

## 8. Variability Analysis

Standard deviation and Interquartile Range (IQR) provide measures of feature spread. `SalePrice` exhibits a high standard deviation of **$79,886.69** and an IQR of **$84,000.00**, reflecting significant price dispersion across different housing tiers.

## 9. Skewness Analysis

Skewness values quantify asymmetry. Features such as `Lot Area` (skew = **12.78**), `SalePrice` (skew = **1.74**), `1st Flr SF` (skew = **1.47**), and `Gr Liv Area` (skew = **1.27**) display high positive skewness, indicating a need for power or log transformations during feature engineering.

In [8]:
skew_df = pd.DataFrame({
    "Feature": num_cols,
    "Skewness": [round(df[c].skew(), 3) for c in num_cols]
}).sort_values(by="Skewness", ascending=False)
display(skew_df.head(10))

## 10. Correlation Analysis

Pearson correlation coefficients measure linear relationships with `SalePrice`.

In [9]:
corr_series = df.select_dtypes(include=np.number).corr()["SalePrice"].drop(["SalePrice", "Order", "PID"]).sort_values(ascending=False)
top_10_corr = pd.DataFrame({
    "Feature": corr_series.head(10).index,
    "Pearson Correlation": corr_series.head(10).values.round(3)
})
display(top_10_corr)

# Plot top correlations
plt.figure(figsize=(10, 5))
sns.barplot(data=top_10_corr, x="Pearson Correlation", y="Feature", palette="crest")
plt.title("Top 10 Numerical Features Correlated with SalePrice", fontsize=14, fontweight="bold", pad=12)
plt.xlabel("Pearson Correlation Coefficient", fontsize=12)
plt.ylabel("Feature", fontsize=12)
plt.tight_layout()
plt.savefig("Figures/SalePrice_Correlation.png")
plt.show()

## 11. Important Statistical Relationships

Visualizing key predictors against `SalePrice`.

In [10]:
# Gr Liv Area vs SalePrice
plt.figure(figsize=(9, 5))
sns.regplot(data=df, x="Gr Liv Area", y="SalePrice", color="#1f77b4", scatter_kws={"alpha":0.4}, line_kws={"color":"darkred"})
plt.title("Above Ground Living Area (Gr Liv Area) vs. SalePrice", fontsize=13, fontweight="bold")
plt.xlabel("Gr Liv Area (sq ft)")
plt.ylabel("Sale Price ($)")
plt.tight_layout()
plt.savefig("Figures/GrLivArea_vs_SalePrice.png")
plt.show()

# Overall Qual vs SalePrice
plt.figure(figsize=(9, 5))
sns.boxplot(data=df, x="Overall Qual", y="SalePrice", palette="Blues_r")
plt.title("Overall Quality Rating vs. SalePrice", fontsize=13, fontweight="bold")
plt.xlabel("Overall Quality Rating")
plt.ylabel("Sale Price ($)")
plt.tight_layout()
plt.savefig("Figures/OverallQual_vs_SalePrice.png")
plt.show()

## 12. Initial Statistical Findings

Based on empirical analysis of the 2,930 property records:
1. **SalePrice Skewness:** `SalePrice` is strongly right-skewed (skewness = **1.74**) with a mean of **$180,796.06** and a median of **$160,000.00**.
2. **Primary Driver:** `Overall Qual` has the highest positive correlation with `SalePrice` ($r = **0.799**$), demonstrating exponential price increases at ratings $\ge 8$.
3. **Living Space Impact:** `Gr Liv Area` is the strongest continuous spatial predictor ($r = **0.707**$), showing a clear positive linear trend.
4. **Garage Capacity & Area:** `Garage Cars` ($r = **0.648**$) and `Garage Area` ($r = **0.640**$) correlate strongly with higher market values.
5. **Basement & 1st Floor Area:** `Total Bsmt SF` ($r = **0.633**$) and `1st Flr SF` ($r = **0.622**$) are major structural determinants.
6. **Property Age Factor:** `Year Built` ($r = **0.558**$) demonstrates that newer construction commands significant market premiums.
7. **Complete Data Integrity:** The dataset contains exactly **2,930 rows, 82 columns, 0 missing values, and 0 duplicate rows** following Week 3 cleaning.

## 13. Conclusion

The Week 5 Data Audit confirms that the cleaned Ames Housing dataset is robust, complete, and free of missing entries. The statistical findings clearly highlight key target-feature relationships and skewness characteristics. This empirical groundwork prepares the dataset for **Week 6 Feature Engineering**, where scaling, normalization (`log1p`), and categorical encoding will be performed.